## Scenario: Caregiver Phone Matches Person Related To Policyholder

**Description:** A caregiver (ICP) billing on an active claim shares a phone number with a person who is related to the claim's policyholder (via the person-person graph), and that related person is not the caregiver or the policyholder themselves. This suggests the caregiver may be a hidden associate of the insured's family.

In [ ]:
ENGINE_CATALOG = dbutils.widgets.get("ENGINE_CATALOG")
ENGINE_SCHEMA = dbutils.widgets.get("ENGINE_SCHEMA")

GRAPH_CATALOG = dbutils.widgets.get("GRAPH_CATALOG")
GRAPH_SCHEMA = dbutils.widgets.get("GRAPH_SCHEMA")

In [ ]:
%sql

DECLARE execDatetime TIMESTAMP = GETDATE();

In [ ]:
# Scenario is not yet registered in T_NOVEL_SCENARIO; skip during testing.
# df = spark.sql(f"""
#   SELECT 
#     NOVEL_SCENARIO_ID 
#   FROM
#     {ENGINE_CATALOG}.{ENGINE_SCHEMA}.T_NOVEL_SCENARIO 
#   WHERE 
#     NOTEBOOK_NAME = 'Scenario_CaregiverPhoneMatchRelatedPerson'
# """)
#
# novelScenarioId = df.collect()[0][0]
# print(f"Novel scenario ID: {novelScenarioId}")

## Parameters

In [ ]:
# Only consider caregivers who billed on this claim within the lookback window
icpInvoiceNumDaysLookback = 365
# ...and whose most recent care activity on the claim is within this cutoff
icpLatestCareNumDaysCutoff = 90
# Drop shared phones linked to this many or more distinct persons (call centers, etc.)
sharedPhoneMaxDistinctPersons = 5

## Active claims

In [ ]:
spark.sql(f"""
SELECT * FROM {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_CLAIM
WHERE CLAIM_STATUS_CODE IN ('Active', 'ASWP', 'Benefit Period', 'Qualification Period')
""").createOrReplaceTempView("active_claims")

## Clean person phones (drop shared/hot phones)

In [ ]:
spark.sql(f"""
WITH hot_phones AS (
  SELECT RES_PHONE_ID
  FROM {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON_PHONE_CROSSWALK
  GROUP BY RES_PHONE_ID
  HAVING COUNT(DISTINCT RES_PERSON_ID) >= {sharedPhoneMaxDistinctPersons}
)
SELECT ph.RES_PERSON_ID, ph.RES_PHONE_ID
FROM {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON_PHONE_CROSSWALK ph
LEFT ANTI JOIN hot_phones h ON h.RES_PHONE_ID = ph.RES_PHONE_ID
""").createOrReplaceTempView("clean_person_phones")

## Policyholders on active claims

In [ ]:
spark.sql(f"""
SELECT DISTINCT
  c.CLAIM_ID,
  c.CLAIM_NUMBER,
  c.POLICY_NUMBER,
  ppc.RES_PERSON_ID AS POLICYHOLDER_ID
FROM active_claims c
JOIN {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON_POLICY_CROSSWALK ppc
  ON ppc.POLICY_NUMBER = c.POLICY_NUMBER
  AND ppc.EDGE_NAME = 'IS_COVERED_BY'
""").createOrReplaceTempView("active_claim_policyholders")

## Persons related to the policyholder

In [ ]:
spark.sql(f"""
SELECT DISTINCT
  p.CLAIM_ID,
  p.CLAIM_NUMBER,
  p.POLICY_NUMBER,
  p.POLICYHOLDER_ID,
  rppc.RES_PERSON_ID_TGT AS RELATED_PERSON_ID,
  rppc.EDGE_NAME AS RELATIONSHIP_TYPE
FROM active_claim_policyholders p
JOIN {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON_PERSON_CROSSWALK rppc
  ON rppc.RES_PERSON_ID_SRC = p.POLICYHOLDER_ID
  AND rppc.RES_PERSON_ID_SRC != rppc.RES_PERSON_ID_TGT
""").createOrReplaceTempView("policyholder_related_persons")

## Phones of related persons

In [ ]:
spark.sql(f"""
SELECT DISTINCT
  rp.CLAIM_ID,
  rp.CLAIM_NUMBER,
  rp.POLICY_NUMBER,
  rp.POLICYHOLDER_ID,
  rp.RELATED_PERSON_ID,
  rp.RELATIONSHIP_TYPE,
  cph.RES_PHONE_ID
FROM policyholder_related_persons rp
JOIN clean_person_phones cph
  ON cph.RES_PERSON_ID = rp.RELATED_PERSON_ID
""").createOrReplaceTempView("related_person_phones")

## Caregivers (ICPs) on active claims

In [ ]:
spark.sql(f"""
SELECT
  a.CLAIM_ID,
  c.CLAIM_NUMBER,
  b.RES_PERSON_ID AS CAREGIVER_ID,
  MAX(a.INVOICE_SERVICE_END_DATE) AS CARE_END_DATE
FROM
  {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_INVOICE a
JOIN
  {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON_INVOICE_CROSSWALK b
  ON a.NORM_INVOICE_ID = b.NORM_INVOICE_ID
  AND b.EDGE_NAME = 'PROVIDED_CARE_ON_INVOICE'
JOIN
  active_claims c
  ON a.CLAIM_ID = c.CLAIM_ID
WHERE
  a.INVOICE_SERVICE_END_DATE >= DATE_SUB(GETDATE(), {icpInvoiceNumDaysLookback})
GROUP BY
  a.CLAIM_ID, c.CLAIM_NUMBER, b.RES_PERSON_ID
HAVING
  MAX(a.INVOICE_SERVICE_END_DATE) >= DATE_SUB(GETDATE(), {icpLatestCareNumDaysCutoff})
""").createOrReplaceTempView("active_claim_caregivers")

## Phones of caregivers

In [ ]:
spark.sql(f"""
SELECT DISTINCT
  c.CLAIM_ID,
  c.CLAIM_NUMBER,
  c.CAREGIVER_ID,
  c.CARE_END_DATE,
  cph.RES_PHONE_ID
FROM active_claim_caregivers c
JOIN clean_person_phones cph
  ON cph.RES_PERSON_ID = c.CAREGIVER_ID
""").createOrReplaceTempView("caregiver_phones")

## Invoice totals (per claim and per caregiver)

In [ ]:
spark.sql(f"""
SELECT
  a.CLAIM_ID,
  SUM(a.INVOICE_CHARGE_AMT) AS TOTAL_CHARGE_AMT,
  SUM(a.INVOICE_PAY_AMT)    AS TOTAL_PAY_AMT
FROM {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_INVOICE a
GROUP BY a.CLAIM_ID
""").createOrReplaceTempView("claim_invoice_totals")

In [ ]:
spark.sql(f"""
SELECT
  b.RES_PERSON_ID AS CAREGIVER_ID,
  a.CLAIM_ID,
  COUNT(DISTINCT a.NORM_INVOICE_ID) AS N_INVOICES_BY_ICP_ON_CLAIM,
  SUM(a.INVOICE_CHARGE_AMT) AS TOTAL_CHARGE_BY_ICP_ON_CLAIM,
  SUM(a.INVOICE_PAY_AMT)    AS TOTAL_PAY_TO_ICP_ON_CLAIM
FROM {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_NORM_INVOICE a
JOIN {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON_INVOICE_CROSSWALK b
  ON a.NORM_INVOICE_ID = b.NORM_INVOICE_ID
  AND b.EDGE_NAME = 'PROVIDED_CARE_ON_INVOICE'
GROUP BY b.RES_PERSON_ID, a.CLAIM_ID
""").createOrReplaceTempView("icp_invoice_totals")

## Flagged claims: caregiver shares a phone with someone related to the policyholder on the same claim

In [ ]:
spark.sql(f"""
SELECT
  cp.CLAIM_ID,
  cp.CLAIM_NUMBER,
  cp.CAREGIVER_ID AS PROVIDER_ID,
  CONCAT(cg.FIRST_NAME, ' ', cg.LAST_NAME) AS PROVIDER_NAME,
  'ICP' AS PROVIDER_TYPE,
  rp.POLICYHOLDER_ID,
  CONCAT(ph.FIRST_NAME, ' ', ph.LAST_NAME) AS POLICYHOLDER_NAME,
  rp.RELATED_PERSON_ID,
  CONCAT(rl.FIRST_NAME, ' ', rl.LAST_NAME) AS RELATED_PERSON_NAME,
  rp.RELATIONSHIP_TYPE,
  cp.RES_PHONE_ID AS SHARED_PHONE_ID,
  CASE WHEN UPPER(TRIM(ph.LAST_NAME)) = UPPER(TRIM(cg.LAST_NAME)) THEN 1 ELSE 0 END AS SAME_LAST_NAME_IND,
  cp.CARE_END_DATE,
  icp_tot.N_INVOICES_BY_ICP_ON_CLAIM,
  icp_tot.TOTAL_CHARGE_BY_ICP_ON_CLAIM,
  icp_tot.TOTAL_PAY_TO_ICP_ON_CLAIM,
  cl_tot.TOTAL_CHARGE_AMT,
  cl_tot.TOTAL_PAY_AMT
FROM caregiver_phones cp
JOIN related_person_phones rp
  ON rp.CLAIM_ID = cp.CLAIM_ID
  AND rp.RES_PHONE_ID = cp.RES_PHONE_ID
JOIN {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON cg
  ON cg.RES_PERSON_ID = cp.CAREGIVER_ID
JOIN {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON ph
  ON ph.RES_PERSON_ID = rp.POLICYHOLDER_ID
JOIN {GRAPH_CATALOG}.{GRAPH_SCHEMA}.T_RESOLVED_PERSON rl
  ON rl.RES_PERSON_ID = rp.RELATED_PERSON_ID
LEFT JOIN claim_invoice_totals cl_tot
  ON cl_tot.CLAIM_ID = cp.CLAIM_ID
LEFT JOIN icp_invoice_totals icp_tot
  ON icp_tot.CLAIM_ID = cp.CLAIM_ID
  AND icp_tot.CAREGIVER_ID = cp.CAREGIVER_ID
WHERE cp.CAREGIVER_ID != rp.RELATED_PERSON_ID
  AND cp.CAREGIVER_ID != rp.POLICYHOLDER_ID
""").createOrReplaceTempView("flagged_claims")

## Trigger counts

In [ ]:
flagged = spark.table("flagged_claims").cache()

n_rows        = flagged.count()
n_claims      = flagged.select("CLAIM_ID").distinct().count()
n_caregivers  = flagged.select("PROVIDER_ID").distinct().count()
n_policyhldrs = flagged.select("POLICYHOLDER_ID").distinct().count()

print(f"Flagged rows                : {n_rows:,}")
print(f"Distinct trigger claims     : {n_claims:,}")
print(f"Distinct flagged caregivers : {n_caregivers:,}")
print(f"Distinct policyholders      : {n_policyhldrs:,}")

## Preview / browse full result

In [ ]:
# Use the Databricks table UI to sort/filter and click 'Download' for CSV/Excel.
display(flagged)

In [ ]:
# Engine table does not exist yet; skip during testing.
# spark.sql(f"""
#     INSERT INTO {ENGINE_CATALOG}.{ENGINE_SCHEMA}.T_SCENARIO_CAREGIVER_PHONE_MATCH_RELATED_PERSON_DETAIL
#     SELECT 
#       *,
#       GETDATE() AS FEATURE_DATETIME
#     FROM flagged_claims;
# """)